# Agente ReAct: Consultor de Continuidad

## Objetivos de Aprendizaje

- Implementar el patrón ReAct (Reasoning + Acting) con LangChain
- Definir herramientas personalizadas con el decorador `@tool`
- Construir un agente que encadena múltiples herramientas para analizar consistencia narrativa
- Observar el ciclo de razonamiento del agente paso a paso

In [ ]:
!pip install langchain langchain-openai langchain-community python-dotenv

## 1. Configuración

El agente usa `gpt-4o` con `temperature=0.1` para razonamiento determinista.

In [ ]:
import os
import re
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain.tools import tool
from langchain import hub
from IPython.display import display, Markdown

load_dotenv()

llm = ChatOpenAI(
    base_url=os.getenv("GITHUB_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o",
    temperature=0.1,
)

print("✓ LLM configurado: gpt-4o")

## 2. Base de Conocimiento

Cargamos los fragmentos del lore y definimos la función de búsqueda léxica que usan las herramientas.

In [ ]:
from lore_database import get_all_fragments, get_fragments_by_category, LoreFragment

fragments = get_all_fragments()
print(f"✓ {len(fragments)} fragmentos de lore cargados")

def _buscar(query, frags, top_k=3):
    query_words = set(re.sub(r"[^\w\s]", "", query.lower()).split())
    scored = []
    for frag in frags:
        text_words = set(re.sub(r"[^\w\s]", "", frag.text.lower()).split())
        overlap = len(query_words & (text_words | set(frag.tags)))
        if overlap > 0:
            scored.append((overlap, frag.text))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [t for _, t in scored[:top_k]]

print("✓ Funcion de busqueda lexica definida")

## 3. Herramientas del Agente

El agente dispone de tres herramientas. El decorador `@tool` convierte una función Python
en una herramienta que el LLM puede seleccionar y ejecutar automáticamente.

In [ ]:
@tool
def search_lore(query: str) -> str:
    """Busca informacion en el lore canonico de Transformers. Usa para verificar hechos generales."""
    results = _buscar(query, get_all_fragments(), top_k=3)
    if not results:
        return "No se encontraron resultados para esa consulta."
    out = f"Lore recuperado para '{query}':\n\n"
    for i, t in enumerate(results, 1):
        out += f"[{i}] {t}\n\n"
    return out.strip()


@tool
def validate_character(character_name: str) -> str:
    """Verifica el estado canonico de un personaje: apariciones, habilidades, muerte, afiliacion."""
    frags = get_fragments_by_category("PERSONAJE")
    name_lower = character_name.lower()
    relevant = [f.text for f in frags if name_lower in f.text.lower() or any(name_lower in tag for tag in f.tags)]
    if not relevant:
        relevant = _buscar(character_name, get_all_fragments(), top_k=2)
    if not relevant:
        return f"Personaje '{character_name}' no encontrado en el lore."
    return f"Estado canonico de '{character_name}':\n\n" + "\n\n".join(f"- {t}" for t in relevant)


@tool
def check_timeline(year_and_event: str) -> str:
    """Valida si un evento es coherente con la linea temporal de la saga. Formato: '2009 - descripcion del evento'."""
    frags = get_fragments_by_category("TIMELINE") + get_fragments_by_category("EVENTO")
    results = _buscar(year_and_event, frags, top_k=4)
    if not results:
        return "No se encontraron eventos canonicos para ese periodo."
    return f"Timeline canonico para '{year_and_event}':\n\n" + "\n\n".join(f"[{i+1}] {t}" for i, t in enumerate(results))


tools = [search_lore, validate_character, check_timeline]
print(f"✓ {len(tools)} herramientas definidas: {[t.name for t in tools]}")

## 4. Construcción del Agente

Usamos `create_openai_tools_agent` con el prompt de LangChain Hub y `AgentExecutor` para correr el ciclo ReAct.

In [ ]:
prompt = hub.pull("hwchase17/openai-tools-agent")

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=6,
    handle_parsing_errors=True,
    return_intermediate_steps=True,
)

print("✓ Agente ReAct construido")

## 5. Prueba 1 — Inconsistencia de Personaje

In [ ]:
fragmento_1 = (
    "Escena 3 - 2007: Bumblebee se gira hacia Sam y dice claramente: "
    "'Sam, debes proteger el AllSpark. Confia en mi.'"
)

result_1 = agent_executor.invoke({"input": fragmento_1})

display(Markdown("---\n**Respuesta del agente:**"))
display(Markdown(result_1["output"]))

## 6. Prueba 2 — Inconsistencia Temporal

In [ ]:
fragmento_2 = (
    "2009. Sam Witwicky y Cade Yeager trabajan juntos en el cuartel de NEST "
    "para analizar los fragmentos del AllSpark."
)

result_2 = agent_executor.invoke({"input": fragmento_2})

display(Markdown("---\n**Respuesta del agente:**"))
display(Markdown(result_2["output"]))

## 7. Prueba 3 — Caso Consistente

In [ ]:
fragmento_3 = (
    "Cinco anos despues de Chicago, Cade Yeager encuentra un camion oxidado en Texas. "
    "Al repararlo descubre que es Optimus Prime."
)

result_3 = agent_executor.invoke({"input": fragmento_3})

display(Markdown("---\n**Respuesta del agente:**"))
display(Markdown(result_3["output"]))

## Conclusiones

- El decorador `@tool` convierte funciones Python en herramientas que el LLM selecciona y ejecuta por sí solo
- Con `verbose=True` se puede ver el ciclo completo: qué herramienta elige, qué input le pasa, qué observa
- El agente encadena hasta 6 iteraciones, lo que permite cruzar personaje + timeline en una sola consulta